## Modelowanie układów przepływowych - ćwiczenia 
#### (budowa solwera równań płytkiej wody na bazie pakietu PyMPDATA)

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip --quiet install open-atmos-jupyter-utils
    from open_atmos_jupyter_utils import pip_install_on_colab
    pip_install_on_colab('PyMPDATA-examples')




### 0. potrzebne pakiety Pythona

In [2]:
import numpy as np
from matplotlib import pyplot
from open_atmos_jupyter_utils import show_plot, show_anim
from PyMPDATA import ScalarField, Solver, Stepper, VectorField, Options, boundary_conditions

### 1. solwer  zbudowany na bazie PyMPDATA

In [3]:


class ShallowWaterEquationsIntegrator:
    def __init__(self, *, h_initial: np.ndarray, options: Options = None, bathymetry: np.ndarray):
        """ initializes the solvers for a given initial condition of `h` assuming zero momenta at t=0 """
        self.bathymetry = bathymetry
        options = options or Options(nonoscillatory=True, infinite_gauge=True)
        X, Y, grid = 0, 1, h_initial.shape
        stepper = Stepper(options=options, grid=grid , n_threads=1)
        kwargs = {
            'boundary_conditions': [boundary_conditions.Constant(value=0)] * len(grid),
            'halo': options.n_halo,
        }
        advectees = {
            "h": ScalarField(h_initial, **kwargs),
            "uh": ScalarField(np.zeros(grid), **kwargs),
            "vh": ScalarField(np.zeros(grid), **kwargs),
        }
        self.advector = VectorField((
                np.zeros((grid[X] + 1, grid[Y])),
                np.zeros((grid[X], grid[Y] + 1))
            ), **kwargs
        )
        self.solvers = { k: Solver(stepper, v, self.advector) for k, v in advectees.items() }

    def __getitem__(self, key):
        """ returns `key` advectee field of the current solver state """
        return self.solvers[key].advectee.get()
    
    def _apply_half_rhs(self, *, key, axis, g_times_dt_over_dxy):
        """ applies half of the source term in the given direction """
        self[key][:] -= .5 * g_times_dt_over_dxy * self['h'] * np.gradient(self['h']-self.bathymetry, axis=axis)

    def _update_courant_numbers(self, *, axis, key, mask, dt_over_dxy):
        """ computes the Courant number component from fluid column height and momenta fields """
        velocity = np.where(mask, np.nan, 0)
        momentum = self[key]
        np.divide(momentum, self['h'], where=mask, out=velocity)

        # using slices to ensure views (over copies)
        all = slice(None, None) 
        all_but_last = slice(None, -1)
        all_but_first_and_last = slice(1, -1)

        velocity_at_cell_boundaries = velocity[( 
            (all_but_last, all),
            (all, all_but_last),
        )[axis]] + np.diff(velocity, axis=axis) / 2 
        courant_number = self.advector.get_component(axis)[(
            (all_but_first_and_last, all),
            (all, all_but_first_and_last)
        )[axis]]
        courant_number[:] = velocity_at_cell_boundaries * dt_over_dxy[axis]
        assert np.amax(np.abs(courant_number)) <= 1

    def __call__(self, *, nt: int, g: float, dt_over_dxy: tuple, outfreq: int, eps: float=1e-7):
        """ integrates `nt` timesteps and returns a dictionary of solver states recorded every `outfreq` step[s] """
        output = {k: [] for k in self.solvers.keys()}
        for it in range(nt + 1): 
            if it != 0:
                mask = self['h'] > eps
                for axis, key in enumerate(("uh", "vh")):
                    self._update_courant_numbers(axis=axis, key=key, mask=mask, dt_over_dxy=dt_over_dxy)
                self.solvers["h"].advance(n_steps=1)
                for axis, key in enumerate(("uh", "vh")):
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
                    self.solvers[key].advance(n_steps=1)
                    self._apply_half_rhs(key=key, axis=axis, g_times_dt_over_dxy=g * dt_over_dxy[axis])
            if it % outfreq == 0:
                for key in self.solvers.keys():
                    output[key].append(self[key].copy())
        return output

### 2. symulacje

In [4]:
grid = (50, 170)

y_profile = np.zeros(grid[1])  

y_profile[0:50] = np.linspace(4, 2, 50) 

x_sin = np.linspace(0, 4*np.pi, 100)
sin_wave = -np.sin(x_sin)
y_profile[50:150] = sin_wave * 1 + 2  
y_profile[150:170] = np.linspace(2, 0, 20)

bathymetry = np.tile(y_profile, (grid[0], 1))

h_initial = bathymetry.copy()
h_initial[
    grid[0] // 50:
    grid[0] -1,
    grid[1]//170 -1:
    grid[1] //170 +3
] += .075

output = ShallowWaterEquationsIntegrator(
    h_initial=h_initial, bathymetry=bathymetry
)(

    nt=800, g=10, dt_over_dxy=(.05, .05), outfreq=3
)



In [5]:
grid_2 = (100, 340)

y_profile = np.zeros(grid_2[1])  

y_profile[0:100] = np.linspace(4, 2, 100) 

x_sin = np.linspace(0, 4*np.pi, 200)
sin_wave = -np.sin(x_sin)
y_profile[100:300] = sin_wave * 1 + 2  
y_profile[300:340] = np.linspace(2, 0, 40)

bathymetry_2 = np.tile(y_profile, (grid_2[0], 1))

h_initial_2 = bathymetry_2.copy()
h_initial_2[
    grid_2[0] // 100:
    grid_2[0] -1,
    grid_2[1]//340 -1:
    grid_2[1] //340 +7
] += .025*3

output_2 = ShallowWaterEquationsIntegrator(
    h_initial=h_initial_2 , bathymetry=bathymetry_2
)(

    nt=3200, g=10, dt_over_dxy=(.025, .025), outfreq=3
)


### 3. Tworzenie PDF

In [17]:


    
y_range_meters = 170  
x_range_meters = 40   

y_indices_original = np.arange(grid[1])
y_meters_original = y_indices_original * (y_range_meters / grid[1])
x_indices_original = np.arange(grid[0])
x_meters_original = x_indices_original * (x_range_meters / grid[0])


y_indices_double = np.arange(grid_2[1])
y_meters_double = y_indices_double * (y_range_meters / grid_2[1])
x_indices_double = np.arange(grid_2[0])
x_meters_double = x_indices_double * (x_range_meters / grid_2[0])


# 1

fig1, ax1 = pyplot.subplots(figsize=(4.13, 5.85))  

extent_original = [0, x_range_meters, 0, y_range_meters]
im1 = ax1.imshow(bathymetry.T, origin='lower', aspect='auto', 
                 cmap='terrain', extent=extent_original)
ax1.set_xlabel('x [m]')
ax1.set_ylabel('y [m] ')
pyplot.colorbar(im1, ax=ax1, label='Głębokość (b) [m]')

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig1.savefig('mapa_batymetrii_2d.pdf', bbox_inches='tight')
pyplot.close(fig1)


# 2

fig2, ax2 = pyplot.subplots(figsize=(4.13, 5.85))

mid_x_idx = grid[0] // 2
mid_x_meters = mid_x_idx * (x_range_meters / grid[0])

ax2.plot(y_meters_original, -bathymetry[mid_x_idx, :], 'b-', linewidth=2)
ax2.set_xlabel('y [m] ')
ax2.set_ylabel('Głębokość [m]')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='z = 0 m')
ax2.legend()

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig2.savefig('profil_batymetrii.pdf', bbox_inches='tight')
pyplot.close(fig2)

# 3 
all_zeta = []
for frame in range(len(output['h'])):
    zeta = output['h'][frame] - bathymetry  # ζ = h - b
    all_zeta.append(zeta)

zeta_3d = np.array(all_zeta)
max_zeta_abs = np.max(np.abs(zeta_3d), axis=0)
max_along_x = np.max(max_zeta_abs, axis=0)

fig3, ax3 = pyplot.subplots(figsize=(4.13, 5.85))

ax3.plot(y_meters_original, max_along_x, 'b-', linewidth=2)
ax3.set_xlabel('y [m]')
ax3.set_ylabel('Maksymalne wychylenie fali  [m]')
ax3.grid(True, alpha=0.3)

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig3.savefig('maksymalne_wychylenie_bezwzgledne.pdf', bbox_inches='tight')
pyplot.close(fig3)

# 4 
fig4, ax4 = pyplot.subplots(figsize=(4.13, 5.85))

extent_original = [0, x_range_meters, 0, y_range_meters]
im4 = ax4.imshow(max_zeta_abs.T, origin='lower', aspect='auto', 
                 cmap='viridis', extent=extent_original)
ax4.set_xlabel('x [m] ')
ax4.set_ylabel('y [m] ')
pyplot.colorbar(im4, ax=ax4, label='Maksymalne wychylenie fali  [m]')

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig4.savefig('mapa_maksymalnych_wychylen.pdf', bbox_inches='tight')
pyplot.close(fig4)


# 5


all_zeta_2 = []
for frame in range(len(output_2['h'])):
    zeta = output_2['h'][frame] - bathymetry_2  # ζ = h - b
    all_zeta_2.append(zeta)

zeta_3d_2 = np.array(all_zeta_2)
max_zeta_abs_2 = np.max(np.abs(zeta_3d_2), axis=0)
max_along_x_2 = np.max(max_zeta_abs_2, axis=0)

fig5, ax5 = pyplot.subplots(figsize=(4.13, 5.85))

ax5.plot(y_meters_double, max_along_x_2, 'b-', linewidth=2)
ax5.set_xlabel('y [m] ')
ax5.set_ylabel('Maksymalne wychylenie fali  [m]')
ax5.grid(True, alpha=0.3)

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig5.savefig('maksymalne_wychylenie_podwojona_rozdzielczosc.pdf', bbox_inches='tight')
pyplot.close(fig5)

# 6

fig6, ax6 = pyplot.subplots(figsize=(4.13, 5.85))

mid_x_idx_2 = grid_2[0] // 2
mid_x_meters_2 = mid_x_idx_2 * (x_range_meters / grid_2[0])

ax6.plot(y_meters_double, -bathymetry_2[mid_x_idx_2, :], 'b-', linewidth=2)
ax6.set_xlabel('y [m] ')
ax6.set_ylabel('Głębokość [m]')
ax6.grid(True, alpha=0.3)
ax6.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='z = 0 m')
ax6.legend()

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig6.savefig('profil_batymetrii_podwojona_rozdzielczosc.pdf', bbox_inches='tight')
pyplot.close(fig6)

# 7


fig7, ax7 = pyplot.subplots(figsize=(4.13, 5.85))

extent_double = [0, x_range_meters, 0, y_range_meters]
im7 = ax7.imshow(bathymetry_2.T, origin='lower', aspect='auto', 
                 cmap='terrain', extent=extent_double)
ax7.set_xlabel('x [m] ')
ax7.set_ylabel('y [m] ')
pyplot.colorbar(im7, ax=ax7, label='Głębokość (b) [m]')

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig7.savefig('mapa_batymetrii_2d_podwojona.pdf', bbox_inches='tight')
pyplot.close(fig7)

# 8

fig8, ax8 = pyplot.subplots(figsize=(4.13, 5.85))

extent_double = [0, x_range_meters, 0, y_range_meters]
im8 = ax8.imshow(max_zeta_abs_2.T, origin='lower', aspect='auto', 
                 cmap='viridis', extent=extent_double)
ax8.set_xlabel('x [m] ')
ax8.set_ylabel('y [m] ')
pyplot.colorbar(im8, ax=ax8, label='Maksymalne wychylenie fali  [m]')

pyplot.tight_layout(rect=[0, 0, 1, 0.95])
fig8.savefig('mapa_maksymalnych_wychylen_podwojona.pdf', bbox_inches='tight')
pyplot.close(fig8)

In [15]:
mask_y1_original = (y_meters_original >= 50) & (y_meters_original <= 80)
mask_y1_double = (y_meters_double >= 50) & (y_meters_double <= 80)
    
mask_y2_original = (y_meters_original >= 80) & (y_meters_original <= 130)
mask_y2_double = (y_meters_double >= 80) & (y_meters_double <= 130)

if np.any(mask_y1_original):
    max_zeta_y1_original = np.max(max_along_x[mask_y1_original])
    print(f"ORYGINALNA ROZDZIELCZOŚĆ - 1 wzniesienie:")
    print(f"  Maksymalne wychylenie: {max_zeta_y1_original:.6f} m")

if np.any(mask_y2_original):
    max_zeta_y2_original = np.max(max_along_x[mask_y2_original])
    print(f"ORYGINALNA ROZDZIELCZOŚĆ - 2 wzniesienie:")
    print(f"  Maksymalne wychylenie: {max_zeta_y2_original:.6f} m")

if np.any(mask_y1_double):
    max_zeta_y1_double = np.max(max_along_x_2[mask_y1_double])
    print(f"\nPODWOJONA ROZDZIELCZOŚĆ - 1 wzniesienie:")
    print(f"  Maksymalne wychylenie: {max_zeta_y1_double:.6f} m")

if np.any(mask_y2_double):
    max_zeta_y2_double = np.max(max_along_x_2[mask_y2_double])
    print(f"PODWOJONA ROZDZIELCZOŚĆ - 2 wzniesienie:")
    print(f"  Maksymalne wychylenie: {max_zeta_y2_double:.6f} m")


if max_zeta_y1_original > 0 and max_zeta_y2_original > 0:
    diff_y1 = abs(max_zeta_y1_original - max_zeta_y2_original)
    diff_pct_y1 = (diff_y1 / max_zeta_y1_original) * 100
    print(f"\nOryginalna rodzielczość ")
    print(f"  Różnica bezwzględna: {diff_y1:.6f} m")
    print(f"  Różnica względna: {diff_pct_y1:.2f}%")

if max_zeta_y1_double > 0 and max_zeta_y2_double > 0:
    diff_y2 = abs(max_zeta_y1_double - max_zeta_y2_double)
    diff_pct_y2 = (diff_y2 / max_zeta_y1_double) * 100
    print(f"\nPodwojona rozdzielczność")
    print(f"  Różnica bezwzględna: {diff_y2:.6f} m")
    print(f"  Różnica względna: {diff_pct_y2:.2f}%")

ORYGINALNA ROZDZIELCZOŚĆ - 1 wzniesienie:
  Maksymalne wychylenie: 0.050437 m
ORYGINALNA ROZDZIELCZOŚĆ - 2 wzniesienie:
  Maksymalne wychylenie: 0.040397 m

PODWOJONA ROZDZIELCZOŚĆ - 1 wzniesienie:
  Maksymalne wychylenie: 0.064408 m
PODWOJONA ROZDZIELCZOŚĆ - 2 wzniesienie:
  Maksymalne wychylenie: 0.058783 m

Oryginalna rodzielczość 
  Różnica bezwzględna: 0.010040 m
  Różnica względna: 19.91%

Podwojona rozdzielczność
  Różnica bezwzględna: 0.005626 m
  Różnica względna: 8.73%
